<a href="https://colab.research.google.com/github/AditiNayak-S/MLT3011_Cloud-Infrastructure-Failure-Prediction/blob/main/notebooks/Machine_LearningTechniques_Lab6_Profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Techniques

## Lab 6: Random Forest and AdaBoost Classification


### Lab Objectives

- Explore Dataset B and identify a suitable dataset for ensemble learning.
- Implement Random Forest and AdaBoost classifiers.
- Compare the predictive performance of both ensemble models.
- Evaluate the classifiers using multiple performance metrics.
- Interpret the results obtained from ensemble learning techniques.

## Importing Required Libraries

The following libraries are required for dataset exploration, preprocessing, ensemble learning, visualization, and model evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
import glob
import zipfile

from google.colab import files

plt.style.use("dark_background")

plt.rcParams["figure.figsize"] = (8,5)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.grid"] = True

print("Libraries imported successfully.")

Libraries imported successfully.


## Uploading Dataset B

Dataset B is uploaded in ZIP format and extracted into the Google Colab environment. The folder structure is then explored to identify the available datasets for ensemble learning.

In [ ]:
uploaded = files.upload()

Saving CIFPS - DATASET B.zip to CIFPS - DATASET B.zip


In [ ]:
uploaded_files = list(uploaded.keys())

for file in uploaded_files:
    print(file)

CIFPS - DATASET B.zip


In [ ]:
for file in uploaded_files:
    with zipfile.ZipFile(file, "r") as zip_ref:
        zip_ref.extractall()

print("Dataset extracted successfully.")

Dataset extracted successfully.


In [ ]:
print("Current Working Directory:\n")
print(os.getcwd())

print("DATASET B DIRECTORY STRUCTURE")


for root, dirs, files in os.walk("."):
    level = root.replace(".", "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    sub_indent = "    " * (level + 1)

    for file in sorted(files):
        print(f"{sub_indent}{file}")

Current Working Directory:

/content
DATASET B DIRECTORY STRUCTURE
./
    CIFPS - DATASET B.zip
    .config/
        .last_opt_in_prompt.yaml
        .last_survey_prompt.yaml
        .last_update_check.json
        active_config
        config_sentinel
        default_configs.db
        gce
        hidden_gcloud_config_universe_descriptor_data_cache_configs.db
        logs/
            2026.06.04/
                13.31.42.499627.log
                13.32.02.654775.log
                13.32.18.735754.log
                13.32.21.210570.log
                13.32.38.346437.log
                13.32.39.344962.log
        configurations/
            config_default
    Failure-Dataset-OpenStack-main/
        LICENSE
        README.md
        DEPL/
            Failure_Labels.txt
            LCS_with_VMM.tsv
            SEQ.tsv
        NET/
            Failure_Labels.txt
            LCS_with_VMM.tsv
            SEQ.tsv
        STO/
            Failure_Labels.txt
            LCS_with_VMM.tsv
  

In [ ]:
all_files = []

for root, dirs, files in os.walk("."):
    for file in files:
        full_path = os.path.join(root, file)
        all_files.append(full_path)

files_df = pd.DataFrame({
    "Available Files": sorted(all_files)
})

display(files_df)

,Available Files
0,./.config/.last_opt_in_prompt.yaml
1,./.config/.last_survey_prompt.yaml
2,./.config/.last_update_check.json
3,./.config/active_config
4,./.config/config_sentinel
5,./.config/configurations/config_default
6,./.config/default_configs.db
7,./.config/gce
8,./.config/hidden_gcloud_config_universe_descri...
9,./.config/logs/2026.06.04/13.31.42.499627.log


In [ ]:
import os

base_path = "Failure-Dataset-OpenStack-main"

for folder in sorted(os.listdir(base_path)):
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):


        print(folder)


        for file in sorted(os.listdir(folder_path)):
            print(file)

DEPL
Failure_Labels.txt
LCS_with_VMM.tsv
SEQ.tsv
NET
Failure_Labels.txt
LCS_with_VMM.tsv
SEQ.tsv
STO
Failure_Labels.txt
LCS_with_VMM.tsv
SEQ.tsv


In [ ]:
import glob

tsv_files = glob.glob(
    "Failure-Dataset-OpenStack-main/**/*.tsv",
    recursive=True
)

print("TSV Files Found:\n")

for file in sorted(tsv_files):
    print(file)

TSV Files Found:

Failure-Dataset-OpenStack-main/DEPL/LCS_with_VMM.tsv
Failure-Dataset-OpenStack-main/DEPL/SEQ.tsv
Failure-Dataset-OpenStack-main/NET/LCS_with_VMM.tsv
Failure-Dataset-OpenStack-main/NET/SEQ.tsv
Failure-Dataset-OpenStack-main/STO/LCS_with_VMM.tsv
Failure-Dataset-OpenStack-main/STO/SEQ.tsv


In [ ]:
for file in sorted(tsv_files):

    print("\n" + "="*80)
    print(os.path.basename(file))
    print("="*80)

    df = pd.read_csv(file, sep="\t")

    print("Shape :", df.shape)

    print("\nColumns:")

    print(df.columns.tolist())

    display(df.head())


LCS_with_VMM.tsv
Shape : (1076, 208)

Columns:
['cinderclient_DELETE_202', 'cinderclient_DELETE_400', 'cinderclient_DELETE_500', 'cinderclient_GET_200', 'cinderclient_GET_400', 'cinderclient_GET_404', 'cinderclient_GET_500', 'cinderclient_POST_200', 'cinderclient_POST_202', 'cinderclient_POST_400', 'cinderclient_POST_404', 'cinderclient_POST_500', 'cinder-scheduler_create_volume', 'cinder-scheduler_create_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_attach_volume', 'cinder-volume.localhost.localdomain@lvm_attach_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_create_volume', 'cinder-volume.localhost.localdomain@lvm_create_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_delete_volume', 'cinder-volume.localhost.localdomain@lvm_delete_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_detach_volume', 'cinder-volume.localhost.localdomain@lvm_detach_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_initialize_connection', 'cinder-volume.localhost.local

,cinderclient_DELETE_202,cinderclient_DELETE_400,cinderclient_DELETE_500,cinderclient_GET_200,cinderclient_GET_400,cinderclient_GET_404,cinderclient_GET_500,cinderclient_POST_200,cinderclient_POST_202,cinderclient_POST_400,...,q-plugin_get_ports.1,q-plugin_release_dhcp_port.1,q-plugin_update_device_list.1,q-plugin_update_device_list_ERROR.1,scheduler_delete_instance_info.1,scheduler_delete_instance_info_ERROR.1,scheduler_select_destinations.1,scheduler_select_destinations_ERROR.1,scheduler_update_instance_info.1,scheduler_update_instance_info_ERROR.1
0,0,0,0,0,0,0,0,0,0,0,...,13,0,1,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,15,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,7,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,7,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,4,0,1,0,0,0,0,0,0,0



SEQ.tsv
Shape : (1076, 104)

Columns:
['cinderclient_DELETE_202', 'cinderclient_DELETE_400', 'cinderclient_DELETE_500', 'cinderclient_GET_200', 'cinderclient_GET_400', 'cinderclient_GET_404', 'cinderclient_GET_500', 'cinderclient_POST_200', 'cinderclient_POST_202', 'cinderclient_POST_400', 'cinderclient_POST_404', 'cinderclient_POST_500', 'cinder-scheduler_create_volume', 'cinder-scheduler_create_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_attach_volume', 'cinder-volume.localhost.localdomain@lvm_attach_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_create_volume', 'cinder-volume.localhost.localdomain@lvm_create_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_delete_volume', 'cinder-volume.localhost.localdomain@lvm_delete_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_detach_volume', 'cinder-volume.localhost.localdomain@lvm_detach_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_initialize_connection', 'cinder-volume.localhost.localdomain@lv

,cinderclient_DELETE_202,cinderclient_DELETE_400,cinderclient_DELETE_500,cinderclient_GET_200,cinderclient_GET_400,cinderclient_GET_404,cinderclient_GET_500,cinderclient_POST_200,cinderclient_POST_202,cinderclient_POST_400,...,q-plugin_get_ports,q-plugin_release_dhcp_port,q-plugin_update_device_list,q-plugin_update_device_list_ERROR,scheduler_delete_instance_info,scheduler_delete_instance_info_ERROR,scheduler_select_destinations,scheduler_select_destinations_ERROR,scheduler_update_instance_info,scheduler_update_instance_info_ERROR
0,1,0,0,10,0,1,0,1,5,0,...,24,2,15,0,1,0,1,0,1,0
1,1,0,0,10,0,1,0,1,5,0,...,26,2,15,0,1,0,1,0,1,0
2,1,0,0,10,0,1,0,1,5,0,...,28,2,15,0,1,0,1,0,1,0
3,1,0,0,10,0,1,0,1,5,0,...,26,3,18,0,1,0,1,0,1,0
4,1,0,0,10,0,1,0,1,5,0,...,28,3,18,0,1,0,1,0,1,0



LCS_with_VMM.tsv
Shape : (561, 118)

Columns:
['compute_attach_interface', 'compute_attach_interface_ERROR', 'compute_build_and_run_instance', 'compute_external_instance_event', 'conductor_build_instances', 'conductor_schedule_and_build_instances', 'conductor_schedule_and_build_instances_ERROR', 'dhcp_agent_network_create_end', 'dhcp_agent_port_create_end', 'dhcp_agent_port_delete_end', 'dhcp_agent_port_update_end', 'dhcp_agent_subnet_create_end', 'ipsec_driver_get_vpn_services_on_host', 'l3_agent_router_deleted', 'l3_agent_routers_updated', 'neutronclient_DELETE_204', 'neutronclient_POST_201', 'neutronclient_PUT_200', 'neutron-vo-Network-1.0_push', 'neutron-vo-Port-1.1_push', 'neutron-vo-SecurityGroup-1.0_push', 'neutron-vo-SecurityGroupRule-1.0_push', 'neutron-vo-Subnet-1.0_push', 'novaclient_GET_200', 'novaclient_GET_400', 'novaclient_GET_404', 'novaclient_GET_500', 'novaclient_POST_200', 'novaclient_POST_202', 'novaclient_POST_400', 'novaclient_POST_403', 'novaclient_POST_404', 'n

...

,compute_attach_interface,compute_attach_interface_ERROR,compute_build_and_run_instance,compute_external_instance_event,conductor_build_instances,conductor_schedule_and_build_instances,conductor_schedule_and_build_instances_ERROR,dhcp_agent_network_create_end,dhcp_agent_port_create_end,dhcp_agent_port_delete_end,...,q-plugin_get_active_networks_info_ERROR,q-plugin_get_network_info,q-plugin_get_network_info_ERROR,q-plugin_get_ports,q-plugin_release_dhcp_port,q-plugin_update_device_list,scheduler_select_destinations,scheduler_select_destinations_ERROR,scheduler_update_instance_info,scheduler_update_instance_info_ERROR
0,1,0,1,7,0,1,0,2,11,3,...,0,4,0,14,0,18,1,0,1,0
1,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,2,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,1,4,0,1,0,2,5,0,...,0,4,0,0,0,8,1,0,1,0
4,1,0,1,7,0,1,0,2,11,3,...,0,4,0,18,0,18,1,0,1,0



LCS_with_VMM.tsv
Shape : (901, 150)

Columns:
['cinderclient_DELETE_202', 'cinderclient_DELETE_400', 'cinderclient_DELETE_500', 'cinderclient_GET_200', 'cinderclient_GET_400', 'cinderclient_GET_500', 'cinderclient_POST_200', 'cinderclient_POST_202', 'cinderclient_POST_400', 'cinderclient_POST_404', 'cinderclient_POST_500', 'cinder-scheduler_create_volume', 'cinder-scheduler_create_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_attach_volume', 'cinder-volume.localhost.localdomain@lvm_attach_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_create_volume', 'cinder-volume.localhost.localdomain@lvm_create_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_delete_volume', 'cinder-volume.localhost.localdomain@lvm_delete_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_detach_volume', 'cinder-volume.localhost.localdomain@lvm_detach_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_initialize_connection', 'cinder-volume.localhost.localdomain@lvm_initialize_con

,cinderclient_DELETE_202,cinderclient_DELETE_400,cinderclient_DELETE_500,cinderclient_GET_200,cinderclient_GET_400,cinderclient_GET_500,cinderclient_POST_200,cinderclient_POST_202,cinderclient_POST_400,cinderclient_POST_404,...,q-plugin_bulk_pull.1,q-plugin_tunnel_sync.1,q-plugin_update_device_list.1,q-plugin_update_device_list_ERROR.1,scheduler_delete_instance_info.1,scheduler_delete_instance_info_ERROR.1,scheduler_select_destinations.1,scheduler_select_destinations_ERROR.1,scheduler_update_instance_info.1,scheduler_update_instance_info_ERROR.1
0,0,0,0,0,0,0,0,0,0,0,...,2,0,5,0,1,0,1,0,0,0
1,0,0,0,0,0,0,0,0,1,0,...,2,0,5,0,1,0,2,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
3,0,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



SEQ.tsv
Shape : (901, 75)

Columns:
['cinderclient_DELETE_202', 'cinderclient_DELETE_400', 'cinderclient_DELETE_500', 'cinderclient_GET_200', 'cinderclient_GET_400', 'cinderclient_GET_500', 'cinderclient_POST_200', 'cinderclient_POST_202', 'cinderclient_POST_400', 'cinderclient_POST_404', 'cinderclient_POST_500', 'cinder-scheduler_create_volume', 'cinder-scheduler_create_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_attach_volume', 'cinder-volume.localhost.localdomain@lvm_attach_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_create_volume', 'cinder-volume.localhost.localdomain@lvm_create_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_delete_volume', 'cinder-volume.localhost.localdomain@lvm_delete_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_detach_volume', 'cinder-volume.localhost.localdomain@lvm_detach_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_initialize_connection', 'cinder-volume.localhost.localdomain@lvm_initialize_connection_ER

,cinderclient_DELETE_202,cinderclient_DELETE_400,cinderclient_DELETE_500,cinderclient_GET_200,cinderclient_GET_400,cinderclient_GET_500,cinderclient_POST_200,cinderclient_POST_202,cinderclient_POST_400,cinderclient_POST_404,...,q-plugin_bulk_pull,q-plugin_tunnel_sync,q-plugin_update_device_list,q-plugin_update_device_list_ERROR,scheduler_delete_instance_info,scheduler_delete_instance_info_ERROR,scheduler_select_destinations,scheduler_select_destinations_ERROR,scheduler_update_instance_info,scheduler_update_instance_info_ERROR
0,1,0,0,6,0,0,1,8,0,0,...,1,0,0,0,0,0,1,0,0,0
1,1,0,0,2,0,0,0,1,1,0,...,1,0,0,0,0,0,0,0,0,0
2,1,0,0,7,0,0,2,8,0,0,...,3,0,6,0,1,0,2,0,2,0
3,1,0,0,7,0,0,2,8,0,0,...,3,0,6,0,1,0,2,0,2,0
4,1,0,0,7,0,0,2,8,0,0,...,3,0,6,0,1,0,2,0,2,0


In [ ]:
summary = []

for file in sorted(tsv_files):

    df = pd.read_csv(file, sep="\t")

    summary.append({
        "Dataset": os.path.basename(os.path.dirname(file)),
        "File": os.path.basename(file),
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Column Names": ", ".join(df.columns[:10]) + (" ..." if len(df.columns) > 10 else "")
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

,Dataset,File,Rows,Columns,Column Names
0,DEPL,LCS_with_VMM.tsv,1076,208,"cinderclient_DELETE_202, cinderclient_DELETE_4..."
1,DEPL,SEQ.tsv,1076,104,"cinderclient_DELETE_202, cinderclient_DELETE_4..."
2,NET,LCS_with_VMM.tsv,561,118,"compute_attach_interface, compute_attach_inter..."
3,NET,SEQ.tsv,561,59,"compute_attach_interface, compute_attach_inter..."
4,STO,LCS_with_VMM.tsv,901,150,"cinderclient_DELETE_202, cinderclient_DELETE_4..."
5,STO,SEQ.tsv,901,75,"cinderclient_DELETE_202, cinderclient_DELETE_4..."


In [ ]:
selected_file = "Failure-Dataset-OpenStack-main/DEPL/LCS_with_VMM.tsv"

df = pd.read_csv(selected_file, sep="\t")

print("Shape :", df.shape)

print("\nColumns:\n")
print(df.columns.tolist())

display(df.head())

display(df.describe(include="all"))

Shape : (1076, 208)

Columns:

['cinderclient_DELETE_202', 'cinderclient_DELETE_400', 'cinderclient_DELETE_500', 'cinderclient_GET_200', 'cinderclient_GET_400', 'cinderclient_GET_404', 'cinderclient_GET_500', 'cinderclient_POST_200', 'cinderclient_POST_202', 'cinderclient_POST_400', 'cinderclient_POST_404', 'cinderclient_POST_500', 'cinder-scheduler_create_volume', 'cinder-scheduler_create_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_attach_volume', 'cinder-volume.localhost.localdomain@lvm_attach_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_create_volume', 'cinder-volume.localhost.localdomain@lvm_create_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_delete_volume', 'cinder-volume.localhost.localdomain@lvm_delete_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_detach_volume', 'cinder-volume.localhost.localdomain@lvm_detach_volume_ERROR', 'cinder-volume.localhost.localdomain@lvm_initialize_connection', 'cinder-volume.localhost.localdomain@lvm_initia

,cinderclient_DELETE_202,cinderclient_DELETE_400,cinderclient_DELETE_500,cinderclient_GET_200,cinderclient_GET_400,cinderclient_GET_404,cinderclient_GET_500,cinderclient_POST_200,cinderclient_POST_202,cinderclient_POST_400,...,q-plugin_get_ports.1,q-plugin_release_dhcp_port.1,q-plugin_update_device_list.1,q-plugin_update_device_list_ERROR.1,scheduler_delete_instance_info.1,scheduler_delete_instance_info_ERROR.1,scheduler_select_destinations.1,scheduler_select_destinations_ERROR.1,scheduler_update_instance_info.1,scheduler_update_instance_info_ERROR.1
0,0,0,0,0,0,0,0,0,0,0,...,13,0,1,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,15,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,7,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,7,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,4,0,1,0,0,0,0,0,0,0


,cinderclient_DELETE_202,cinderclient_DELETE_400,cinderclient_DELETE_500,cinderclient_GET_200,cinderclient_GET_400,cinderclient_GET_404,cinderclient_GET_500,cinderclient_POST_200,cinderclient_POST_202,cinderclient_POST_400,...,q-plugin_get_ports.1,q-plugin_release_dhcp_port.1,q-plugin_update_device_list.1,q-plugin_update_device_list_ERROR.1,scheduler_delete_instance_info.1,scheduler_delete_instance_info_ERROR.1,scheduler_select_destinations.1,scheduler_select_destinations_ERROR.1,scheduler_update_instance_info.1,scheduler_update_instance_info_ERROR.1
count,1076.0,1076.000000,1076.000000,1076.000000,1076.000000,1076.0,1076.000000,1076.0,1076.000000,1076.000000,...,1076.000000,1076.000000,1076.000000,1076.0,1076.000000,1076.0,1076.000000,1076.0,1076.0,1076.0
mean,0.0,0.052974,0.001859,0.000929,0.039963,0.0,0.094796,0.0,0.000929,0.065056,...,11.334572,0.178439,1.478625,0.0,0.227695,0.0,0.045539,0.0,0.0,0.0
std,0.0,0.224086,0.043093,0.030486,0.394475,0.0,0.718092,0.0,0.030486,0.246739,...,13.400366,0.454169,2.087506,0.0,0.419540,0.0,0.208580,0.0,0.0,0.0
min,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
25%,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,...,1.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
50%,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,...,5.000000,0.000000,1.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
75%,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,...,21.000000,0.000000,2.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
max,0.0,1.000000,1.000000,1.000000,7.000000,0.0,8.000000,0.0,1.000000,1.000000,...,42.000000,2.000000,10.000000,0.0,1.000000,0.0,1.000000,0.0,0.0,0.0


In [ ]:
label_file = "Failure-Dataset-OpenStack-main/DEPL/Failure_Labels.txt"

with open(label_file, "r") as file:

    lines = file.readlines()

print("Total Labels :", len(lines))

print("\nFirst 20 Labels:\n")

for label in lines[:20]:
    print(label.strip())

Total Labels : 1076

First 20 Labels:

6
6
6
6
6
6
6
6
6
6
2
2
2
2
2
2
2
2
4
2


In [ ]:
label_file = "Failure-Dataset-OpenStack-main/DEPL/Failure_Labels.txt"

labels = pd.read_csv(label_file, header=None, names=["Label"])

display(labels["Label"].value_counts().sort_index())

,count
Label,
1,224
2,151
3,41
4,69
5,52
6,539
